In [15]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np


In [16]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
939,There are plenty of comments already posted sa...,negative
909,This tender beautifully crafted production del...,positive
169,While this movie did have a few scary moments ...,negative
112,"Mixing small town sheriffs, high-school studen...",negative
888,I was dragged to this movie about four years a...,positive


In [21]:
# data preprocessing

#  Define text preprocessing features
def lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)


def remove_stop_words(text):
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛',"")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise
    


In [22]:
df = normalize_text(df)
df.head()

,review,sentiment
939,plenty comment already posted saying exactly f...,negative
909,tender beautifully crafted production delved d...,positive
169,movie scary moment (great use music film angle...,negative
112,"mixing small town sheriffs, high-school studen...",negative
888,dragged movie four year ago french actress fri...,positive


In [23]:
df['sentiment'].value_counts()

sentiment
negative    263
positive    237
Name: count, dtype: int64

In [26]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [28]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
939,plenty comment already posted saying exactly f...,0
909,tender beautifully crafted production delved d...,1
169,movie scary moment (great use music film angle...,0
112,"mixing small town sheriffs, high-school studen...",0
888,dragged movie four year ago french actress fri...,1


In [29]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [30]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [31]:
X_train,X_test,y_train,y_test = train_test_split(X, y ,test_size=0.25, random_state= 42)

In [35]:
import dagshub
mlflow.set_tracking_uri('https://dagshub.com/dhanushpavann/Capstone-Project.mlflow')
dagshub.init(repo_owner='dhanushpavann', repo_name='Capstone-Project', mlflow=True)

mlflow.set_experiment("logistic Regression Baseline")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

/opt/homebrew/Caskroom/miniconda/base/envs/leo/lib/python3.10/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=77676b18-6f85-4444-a0a9-f78f78d9099a&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c5ffd3caa37bdf66ac8faea557c4126a050b343c75d43c12814639e0d1af992b




Accessing as dhanushpavann

Initialized MLflow to track repo "dhanushpavann/Capstone-Project"

Repository dhanushpavann/Capstone-Project initialized!

2026/10/31 21:27:25 INFO mlflow.tracking.fluent: Experiment with name 'logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/45b17c872dd346afbdc30978d1a7c921', creation_time=1793462245580, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1793462245580, lifecycle_stage='active', name='logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [36]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run.......")

with mlflow.start_run():
    start_time = time.time()

    try:
        logging.info("Logging preprocessing parameters..")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000) # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model")
        model.fit(X_train,y_train)
        logging.info("Model training completed")

        logging.info("Logging model parameters")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making Predictions..")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics..")
        accuracy = accuracy_score(y_test,y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model")
        mlflow.sklearn.log_model(model, "model")


        #Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds. ")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")
    
    except Exception as e:
        logging.error(f"An error occured: {e}, exxc_info=True")

2026-10-31 21:38:59,546 - INFO - Starting MLflow run.......
2026-10-31 21:39:06,562 - INFO - Logging preprocessing parameters..
2026-10-31 21:39:13,112 - INFO - Initializing Logistic Regression model...
2026-10-31 21:39:13,114 - INFO - Fitting the model
2026-10-31 21:39:13,203 - INFO - Model training completed
2026-10-31 21:39:13,204 - INFO - Logging model parameters
2026-10-31 21:39:17,004 - INFO - Making Predictions..
2026-10-31 21:39:17,007 - INFO - Calculating evaluation metrics..
2026-10-31 21:39:17,021 - INFO - Logging evaluation metrics...
2026-10-31 21:39:18,642 - INFO - Saving and logging the model
2026/10/31 21:39:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-10-31 21:39:30,853 - INFO - Model training and logging completed in 24.29 seconds. 
2026-10-31 21:39:30,855 - INFO - Accuracy: 0.696
2026-10-31 21:39:30,855 - INFO - Precision: 0.6842105263157895
2026-10-31 21:39:30,855 - INFO - Recall: 0.6610169491525424
2026-10-31 21:39:

🏃 View run thundering-hawk-101 at: https://dagshub.com/dhanushpavann/Capstone-Project.mlflow/#/experiments/0/runs/ed2be291085643cbbd6f2c27918a289d
🧪 View experiment at: https://dagshub.com/dhanushpavann/Capstone-Project.mlflow/#/experiments/0
